# Prepare 8K midtrain + 16K SFT on CPU

Select **CPU**, turn **Internet on**, attach the tokenizer dataset
`xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb`, and enable the Kaggle secret
`HF_TOKEN` after accepting the official Salesforce xLAM dataset access conditions.

Builds a **600M-token midtrain allocation** (480M documents / 30M reasoning /
90M agents), with 10% preparation headroom. Canonical full histories are saved
before stage selection. SFT has its own 30M reasoning / 60M agent collection
targets and a configurable share of genuinely long (>8K) individual traces;
stage ownership is task-disjoint, so selected SFT traces are not repacked into
midtrain. A trace that has no complete 8K prefix is retained directly for SFT.
Its one-pass training budget is computed after materialization at a separate 1:2
reasoning/agent ratio.

To reuse the unused FineWeb pretrain tail, also attach the pretrain-tokenized
dataset and completed pretrain checkpoint; set both paths below. The checkpoint's
exact shuffle seed/cursor selects unconsumed rows. Leaving both blank streams new
FineWeb-Edu instead (not guaranteed disjoint from pretraining).

Tool observations are preserved. Overlong conversations end at a complete assistant
turn; this retains the original prefix, never a disconnected tail. Overlong single
reasoning answers and Pivot targets that cannot fit are dropped. Canonical JSONL
is retained for later repacking. A stable task hash holds out 2% of trace tasks
before packing. OpenResearcher requires a conservative exact final-answer match to its reference;
completion status alone is not accepted as correctness. This does not verify every action.

Source exhaustion and any unfilled SFT-long quota are reported; the audit stops
before TPU training rather than repeating data, substituting a short trace for a
long quota, or silently reusing a task across stages.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys
os.environ['JAX_PLATFORMS'] = 'cpu'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ['RAYON_NUM_THREADS'] = str(max(1, (os.cpu_count() or 2) - 1))
SOURCE_REF = 'codex/midtrain8k-sft16k'
TOKENIZER_ROOT = Path('/kaggle/input/datasets/xiayicheng3gmailcom/nano-dsv41f-tokenizer-fineweb')
PRETRAIN_CORPUS = ''
PRETRAIN_CHECKPOINT = ''  # directory containing manifest.json or latest.json
MIDTRAIN_TOKENS = 600_000_000
SFT_REASONING_TOKENS = 30_000_000
SFT_AGENT_TOKENS = 60_000_000
SFT_LONG_TOKEN_FRACTION = 0.50
OUTPUT = Path('/kaggle/working/posttrain-corpus-v2')
# Resume CPU preparation from saved outputs by copying posttrain-corpus to OUTPUT.
# Completed source manifests are reused only with identical build settings.
ROOT = Path('/kaggle/working/nano-dsv4.1f')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/xiayicheng3-code/nano-dsv4.1f.git',str(ROOT)],check=True)
if subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip():
    raise RuntimeError('Preserve tracked edits before updating this checkout')
subprocess.run(['git','-C',str(ROOT),'fetch','--depth','1','origin',SOURCE_REF],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','jax[cpu]==0.10.2',
                'datasets>=3.0','tokenizers>=0.21','deepseek-recipe==0.1.1','huggingface_hub'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(ROOT),'--no-deps'],check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
import json
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Enable the HF_TOKEN Kaggle secret for the gated xLAM source') from exc
if not TOKENIZER_ROOT.exists():
    matches = list(Path('/kaggle/input').rglob('nano-dsv41f-tokenizer-fineweb'))
    if len(matches) != 1: raise FileNotFoundError('Attach the tokenizer dataset')
    TOKENIZER_ROOT = matches[0]
files = list(TOKENIZER_ROOT.rglob('tokenizer.json'))
if len(files) != 1: raise ValueError('Expected one tokenizer.json')
TOKENIZER = files[0]
# Check gated access before spending CPU time on the document corpus.
from datasets import load_dataset
xlam_probe = load_dataset('Salesforce/xlam-function-calling-60k',split='train',streaming=True)
next(iter(xlam_probe))
del xlam_probe
print('Official xLAM access verified')
if bool(PRETRAIN_CORPUS) != bool(PRETRAIN_CHECKPOINT):
    raise ValueError('Provide both pretrain corpus and checkpoint, or leave both blank')
command = [sys.executable,'-u','scripts/prepare_posttrain_corpus.py',
    '--tokenizer',str(TOKENIZER),'--output',str(OUTPUT),
    '--midtrain-tokens',str(MIDTRAIN_TOKENS),'--headroom','1.10',
    '--sft-reasoning-tokens',str(SFT_REASONING_TOKENS),
    '--sft-agent-tokens',str(SFT_AGENT_TOKENS),
    '--sft-long-token-fraction',str(SFT_LONG_TOKEN_FRACTION),
    '--tokenize-batch-size','16','--shard-rows','128','--seed','1701']
if PRETRAIN_CORPUS:
    command += ['--pretrain-corpus',PRETRAIN_CORPUS,'--pretrain-checkpoint',PRETRAIN_CHECKPOINT]
# Does not print the token or include it in command arguments.
with Path('/kaggle/working/prepare-posttrain.log').open('a',buffering=1) as log:
    p = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        print(line,end='',flush=True); log.write(line)
    if p.wait(): raise RuntimeError('Corpus preparation failed; inspect prepare-posttrain.log')

In [ ]:
sys.path.insert(0,str(ROOT / 'scripts'))
from posttrain_input import inspect, capacities, sft_budget
_, manifest, fingerprint = inspect(OUTPUT)
print('Corpus checksum:',fingerprint)
for stage in ('midtrain','sft'):
    print(stage, 'train capacity:', capacities(manifest,stage))
    for source in manifest['sources']:
        v = source['views'][stage]['train']
        print(source['source']['key'], {
            k:v.get(k,0) for k in ('records','real_tokens','supervised_tokens','genuine_over_8k_records')})
        if source['exhausted_before_target']:
            print('  Source exhausted before requested quota:',source['collection'])
print('Initial one-pass SFT raw-token budget:',sft_budget(manifest))
overlap = {s['source']['key']: s['selection']['cross_stage_task_overlap'] for s in manifest['sources']
           if s['selection']['cross_stage_task_overlap']}
if overlap: raise RuntimeError(f'Task overlap between stages: {overlap}')
long_shortfalls = {s['source']['key']: s['selection']['shortfall_tokens']['sft_long']
                   for s in manifest['sources'] if s['selection']['shortfall_tokens']['sft_long']}
if long_shortfalls:
    raise RuntimeError(f'SFT long-trace quotas underfilled; inspect source exhaustion before training: {long_shortfalls}')
long_records = sum(s['views']['sft']['train'].get('genuine_over_8k_records',0) for s in manifest['sources'])
if not long_records: raise RuntimeError('No genuine >8K traces survived: inspect source length rejection statistics')
for pool, weight in manifest['pool_mix']['midtrain'].items():
    available = capacities(manifest,'midtrain')[pool]
    required = MIDTRAIN_TOKENS*weight + 4*8192
    if available < required:
        raise RuntimeError(f'{pool} underfilled: {available:,} < {required:,.0f}. Review source audit before training.')
print('Save Version -> Save & Run All. Save the posttrain-corpus output as a Kaggle dataset.')

## Outputs

Attach the saved corpus dataset and your completed pretrain checkpoint to the TPU
notebook. Keep `posttrain_manifest.json`, `tokenizer.json`, `documents/`, `midtrain/`,
`sft/`, and `canonical/` together. Uncompressed numeric shards favor fast CPU writes
and TPU reads; no extra ZIP copy is made. Canonical/source manifests preserve licenses,
provenance, rejection counts and task-split metadata. Saving outputs does not publish
them; choose dataset visibility and attribution when you create the Kaggle dataset.